In [618]:
import numpy as np
from math import comb
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import shap
import pandas as pd
import time
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans




import itertools
import numpy as np
import scipy.special


def weighted_regression_shap(
    model,
    test_instance,
    background_data,
    num_samples="auto",
    normalize_weights=True,
    include_intermediate_factor=True
):
    """
    Compute Shapley values using weighted least-squares regression with explicit subset construction.

    Args:
        model (callable): Prediction function of the trained model.
        test_instance (np.ndarray): Single test instance to explain (1D array).
        background_data (np.ndarray): Background data for feature replacement (2D array).
        num_samples (int or str): Number of subsets to sample. Default is "auto".
        normalize_weights (bool): Normalize kernel weights.
        include_intermediate_factor (bool): Include intermediate factor for kernel weights.

    Returns:
        shap_values (np.ndarray): Array of Shapley values for each feature.
        baseline_prediction (float): Model's average prediction on background data.
    """
    num_features = test_instance.shape[0]

    def shapley_kernel(M, s):
        if s == 0 or s == M:
            return 0
        weight = (M - 1) / (scipy.special.binom(M, s) * s * (M - s))
        if include_intermediate_factor:
            intermediate_factor = 1 / (1 + abs(M / 2 - s))
            weight *= intermediate_factor
        return weight

    # Compute baseline prediction
    baseline_prediction = np.mean(model(background_data))

    # Generate all subsets (powerset)
    subsets = list(itertools.chain.from_iterable(itertools.combinations(range(num_features), r) for r in range(num_features + 1)))

    # Prepare matrix X, weights, and value matrix V
    num_subsets = len(subsets)
    X = np.zeros((num_subsets, num_features + 1))
    X[:, -1] = 1  # Add intercept term
    weights = np.zeros(num_subsets)
    V = np.zeros((num_subsets, num_features))

    for i, subset in enumerate(subsets):
        subset = list(subset)
        V[i, :] = np.mean(background_data, axis=0)  # Start with the baseline
        V[i, subset] = test_instance[subset]  # Replace subset features with test instance values
        X[i, subset] = 1  # Mark active features in the subset
        weights[i] = shapley_kernel(num_features, len(subset))  # Compute kernel weight for subset size

    if normalize_weights:
        weights /= np.sum(weights)  # Normalize weights

    # Compute predictions
    y = model(V)

    # Apply weighted least squares regression
    wsq = np.sqrt(weights)
    X_weighted = wsq[:, None] * X
    y_weighted = wsq * y
    coef = np.linalg.lstsq(X_weighted, y_weighted, rcond=None)[0]

    shap_values = coef[:-1]  # Feature contributions
    return shap_values, coef[-1]






# Generate a regression dataset
X, y = make_regression(n_samples=10000, n_features=10, noise=0.5)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train a RandomForestRegressor
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_train)

# background = shap.sample(X_train, 100)
# background = shap.kmeans(X_train, 10).data

# K-Means Clustering
from sklearn.cluster import KMeans
# Perform K-means clustering
kmeans = KMeans(n_clusters=100, random_state=42)
kmeans.fit(X_train)
# Get cluster centers (centroids)
background = kmeans.cluster_centers_

# Select a test instance
test_instance = X_test[0]

# Compute SHAP values using TreeExplainer
start_time = time.time()
tree_explainer = shap.TreeExplainer(regressor)
tree_shap_values = tree_explainer.shap_values(test_instance.reshape(1, -1))[0]
tree_baseline = float(tree_explainer.expected_value[0])
tree_time = time.time() - start_time

# Compute SHAP values using KernelExplainer
start_time = time.time()
kernel_explainer = shap.KernelExplainer(regressor.predict, background)
kernel_shap_values = kernel_explainer.shap_values(test_instance.reshape(1, -1))[0]
kernel_baseline = float(kernel_explainer.expected_value)
kernel_time = time.time() - start_time

# Compute SHAP values using vectorized_weighted_shap
start_time = time.time()
# background = shap.sample(X_train, 100)
custom_shap_values, custom_baseline = vectorized_weighted_shap(
    model=regressor.predict,
    test_instance=test_instance,
    background_data=background,
    num_samples="auto",
    normalize_weights=True,
    use_random_sampling=True,
    use_paired_sampling=False,
    include_intermediate_factor=True,
    use_random_baseline=True,
    substitute_mean_for_continuous=False,  # New parameter for continuous substitution
    substitute_dummy_flip=False           # New parameter for dummy variable substitution
)

custom_time = time.time() - start_time

# Compute SHAP values using weighted_regression_shap
start_time = time.time()

regression_shap_values, regression_baseline = weighted_regression_shap(
    model=regressor.predict,
    test_instance=test_instance,
    background_data=background,
    num_samples="auto",
    normalize_weights=True,
    include_intermediate_factor=True,
)

regression_time = time.time() - start_time

# Compute MAE for Kernel and Custom with TreeExplainer as the reference
tree_mae = mean_absolute_error(tree_shap_values, tree_shap_values)
kernel_mae = mean_absolute_error(tree_shap_values, kernel_shap_values)
custom_mae = mean_absolute_error(tree_shap_values, custom_shap_values)
regression_mae = mean_absolute_error(tree_shap_values, regression_shap_values)

# Display results
results = pd.DataFrame({
    "Feature": [f"Feature {i+1}" for i in range(X.shape[1])],
    "TreeExplainer SHAP": np.round(tree_shap_values, 4),
    "KernelExplainer SHAP": np.round(kernel_shap_values, 4),
    "Custom SHAP": np.round(custom_shap_values, 4),
    "Regression SHAP": np.round(regression_shap_values, 4),
})

results = pd.concat([
    results,
    pd.DataFrame([
        {"Feature": "Baseline", "TreeExplainer SHAP": np.round(tree_baseline, 4),
         "KernelExplainer SHAP": np.round(kernel_baseline, 4),
         "Custom SHAP": np.round(custom_baseline, 4),
         "Regression SHAP": np.round(regression_baseline, 4)},
        {"Feature": "Sum", "TreeExplainer SHAP": np.round(np.sum(tree_shap_values) + tree_baseline, 4),
         "KernelExplainer SHAP": np.round(np.sum(kernel_shap_values) + kernel_baseline, 4),
         "Custom SHAP": np.round(np.sum(custom_shap_values) + custom_baseline, 4),
         "Regression SHAP": np.round(np.sum(regression_shap_values) + regression_baseline, 4)},
        {"Feature": "Prediction (Test Instance)", "TreeExplainer SHAP": np.round(regressor.predict(test_instance.reshape(1, -1))[0], 4),
         "KernelExplainer SHAP": np.round(regressor.predict(test_instance.reshape(1, -1))[0], 4),
         "Custom SHAP": np.round(regressor.predict(test_instance.reshape(1, -1))[0], 4),
         "Regression SHAP": np.round(regressor.predict(test_instance.reshape(1, -1))[0], 4)},
        {"Feature": "Computation Time (s)", "TreeExplainer SHAP": np.round(tree_time, 4),
         "KernelExplainer SHAP": np.round(kernel_time, 4),
         "Custom SHAP": np.round(custom_time, 4),
         "Regression SHAP": np.round(regression_time, 4)},
        {"Feature": "MAE (vs TreeExplainer)", "TreeExplainer SHAP": np.round(tree_mae, 4),
         "KernelExplainer SHAP": np.round(kernel_mae, 4),
         "Custom SHAP": np.round(custom_mae, 4),
         "Regression SHAP": np.round(regression_mae, 4)},
    ])
])

results

C:\Users\mnoorche\AppData\Local\Temp\ipykernel_34852\470844826.py:609: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  tree_baseline = float(tree_explainer.expected_value)


  0%|          | 0/1 [00:00<?, ?it/s]

,Feature,TreeExplainer SHAP,KernelExplainer SHAP,Custom SHAP,Regression SHAP,Exact Shapley SHAP
0,Feature 1,41.3619,41.0490,38.3522,12760.4117,12760.4117
1,Feature 2,-6.3164,-6.3479,-6.6630,-2130.1651,-2130.1651
2,Feature 3,-43.9518,-44.0539,-43.4554,-10937.6269,-10937.6269
3,Feature 4,-4.7647,-5.1335,-4.6387,-1032.3546,-1032.3546
4,Feature 5,-0.0132,0.1103,0.5894,-235.8719,-235.8719
5,Feature 6,-11.9577,-10.7593,-10.1434,-2202.9492,-2202.9492
6,Feature 7,-58.0380,-54.3327,-57.6408,-8718.5176,-8718.5176
7,Feature 8,-1.2545,-0.9950,-0.3793,-188.7386,-188.7386
8,Feature 9,-2.3063,-2.6111,-1.2027,-198.6389,-198.6389
9,Feature 10,66.1886,66.0871,68.1945,17753.9806,17753.9806
